# INC-001 Data Findings: Detect, Decide, Demonstrate

A compact, reproducible walkthrough of the profiling findings. Each finding connects **how it was detected**, a concrete **scenario**, the approved **handling and why**, and a **before → after** result.

This guide reads raw files without modifying them. Profiling scans are streaming by participant. Production layer functions are used for the solved examples. Raw measurement rows are not displayed.

## Reading this notebook
- **Observed** means measured in the INC-001 profiling run; **decision** means approved in `reports/INC-001/profiling-findings.md`.
- **Implemented** means the measure exists in the current pipeline and has implementation evidence; **open** means no equivalent production handling was found.
- Scenarios display participant/date identifiers and derived values only.
- To keep dependencies and memory modest, source tables use typed Python record lists rather than pandas DataFrames; the 2-million-slot daily grid is represented sparsely as observed keys and missing counts.

## Sections
1. Parameters and paths
2. Shared parsers, validators and rule helpers
3. Load and standardize the four source families
4. Canonical minute grid and local-time normalization
5. Finding registry
6. Tabular checks, T-01–T-03
7. Sleep checks, S-01–S-07
8. Steps and heart-rate checks, M-01–M-11
9. Cross-source checks, X-01–X-03
10. Steps-per-hour walkthrough
11. Duplicate and gap walkthrough
12. Heart-rate range and stuck-sensor walkthrough
13. Solved silver/gold examples
14. Count and regression validation
15. Rule-helper tests

## 1-2. Parameters, paths, and shared utilities

One configuration object controls the study window, local offset, validity bounds, run threshold, and display limits. CSV/JSON readers preserve source path and row position; parsers return normalized values without altering raw fields.

In [8]:
# 1-2. Parameters, paths, parsers, validators, and compact evidence helpers.
from __future__ import annotations

import csv
import json
import re
import sys
from collections import Counter, defaultdict
from dataclasses import dataclass
from datetime import date, datetime, timedelta, timezone
from pathlib import Path
from statistics import median
from typing import Any, Iterable, Mapping, Sequence

ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / 'src' / 'wearable_pipeline').is_dir()),
    None,
)
if ROOT is None:
    raise RuntimeError('Open this notebook from inside the project workspace.')
sys.path.insert(0, str(ROOT / 'src'))

from wearable_pipeline.heart_rate import build_heart_rate_layer
from wearable_pipeline.normalize import parse_iso_date, parse_local_timestamp
from wearable_pipeline.sleep import build_sleep_layer, compute_sleep_metrics
from wearable_pipeline.steps import build_steps_layer


@dataclass(frozen=True)
class ProfileConfig:
    data_dir: Path = ROOT / 'data'
    findings_path: Path = ROOT / 'reports' / 'INC-001' / 'profiling-findings.md'
    study_start: date = date(2026, 1, 8)
    study_days: int = 28
    local_offset_hours: int = -5
    expected_minutes_per_day: int = 1440
    steps_burst_threshold: int = 30
    heart_rate_min: int = 40
    heart_rate_max: int = 200
    stuck_minutes: int = 60
    show_example_limit: int = 3

    @property
    def study_dates(self) -> tuple[date, ...]:
        return tuple(self.study_start + timedelta(days=i) for i in range(self.study_days))


CFG = ProfileConfig()


def read_csv_records(path: Path) -> list[dict[str, str]]:
    with path.open(newline='', encoding='utf-8-sig') as handle:
        return list(csv.DictReader(handle))


def read_json_records(path: Path) -> list[dict[str, Any]]:
    with path.open(encoding='utf-8') as handle:
        payload = json.load(handle)
    if not isinstance(payload, list) or not all(isinstance(item, dict) for item in payload):
        raise ValueError(f'Expected a JSON array of objects: {path}')
    return payload


def local_timestamp(raw: str | None) -> datetime | None:
    return parse_local_timestamp(raw)


def minute_slot(value: datetime) -> datetime:
    return value.replace(second=0, microsecond=0)


def duplicate_groups(records: Iterable[tuple[str, datetime, Any]]) -> dict[tuple[str, datetime], list[Any]]:
    groups: dict[tuple[str, datetime], list[Any]] = defaultdict(list)
    for participant_id, timestamp, value in records:
        groups[(participant_id, minute_slot(timestamp))].append(value)
    return {key: values for key, values in groups.items() if len(values) > 1}


def finding_row(finding_id: str, severity: str, handling: str,
                evidence_count: int, example_ids: Sequence[str]) -> dict[str, Any]:
    return {
        'finding_id': finding_id,
        'severity': severity,
        'handling': handling,
        'evidence_count': evidence_count,
        'example_ids': ', '.join(example_ids[:CFG.show_example_limit]),
    }


def compact_rows(title: str, rows: Sequence[Mapping[str, Any]]) -> None:
    print(f'\n{title}')
    if not rows:
        print('(no rows)')
        return
    columns = list(rows[0])
    print(' | '.join(columns))
    print(' | '.join('---' for _ in columns))
    for row in rows:
        print(' | '.join(str(row.get(column, '')) for column in columns))


ENROLLED = {
    row['participant_id']
    for row in read_csv_records(CFG.data_dir / 'participants' / 'participants.csv')
}
print(f'Workspace: {ROOT}')
print(f'Study window: {CFG.study_start} through {CFG.study_dates[-1]} ({CFG.study_days} days)')
print(f'Enrolled participants: {len(ENROLLED)}; local offset: UTC{CFG.local_offset_hours:+d}:00')

Workspace: /home/michaelmadeira/wearable-health-data-pipeline
Study window: 2026-01-08 through 2026-02-04 (28 days)
Enrolled participants: 50; local offset: UTC-5:00


In [18]:
# Shared provenance and table helpers; record lists keep this guide dependency-light.
import html
from IPython.display import HTML, display


def read_csv_records(path: Path) -> list[dict[str, Any]]:
    with path.open(newline='', encoding='utf-8-sig') as handle:
        rows = list(csv.DictReader(handle))
    relative = path.relative_to(CFG.data_dir).as_posix()
    return [dict(row, source_file=relative, row_index=index)
            for index, row in enumerate(rows, start=2)]


def read_json_records(path: Path) -> list[dict[str, Any]]:
    with path.open(encoding='utf-8') as handle:
        payload = json.load(handle)
    if not isinstance(payload, list) or not all(isinstance(item, dict) for item in payload):
        raise ValueError(f'Expected a JSON array of objects: {path}')
    relative = path.relative_to(CFG.data_dir).as_posix()
    return [dict(item, source_file=relative, row_index=index)
            for index, item in enumerate(payload, start=1)]


def compact_rows(title: str, rows: Sequence[Mapping[str, Any]]) -> None:
    print(title)
    if not rows:
        print('(no rows)')
        return
    columns = list(dict.fromkeys(key for row in rows for key in row))
    header = ''.join(f'<th>{html.escape(str(column))}</th>' for column in columns)
    body = ''.join(
        '<tr>' + ''.join(
            f'<td>{html.escape(str(row.get(column, "")))}</td>' for column in columns
        ) + '</tr>' for row in rows
    )
    display(HTML(
        '<div style="overflow-x:auto"><table><thead><tr>' + header
        + '</tr></thead><tbody>' + body
        + '</tbody></table></div>'
    ))

## 5. Finding registry

The registry below keeps detection logic, observed scenario, approved handling, rationale, solved illustration, and implementation status together. Status is deliberately not inferred from the finding severity.

In [7]:
# 5. Rule registry: check, concrete scenario, approved handling, rationale, solved outcome.
# Status reflects reviewed implementation evidence through INC-005.
FINDINGS = [
    ('T-01', 'Medium', 'survey participant_id not in enrolled IDs', 'P999: 14 daily rows', 'Keep and flag in silver; exclude from enrolled outputs', 'Preserves source evidence without joining an unenrolled person.', 'P999 rows remain auditable; enrolled gold stays at 1,400 rows.', 'Implemented'),
    ('T-02', 'Medium', 'blank firmware_version with parseable device_label', 'P006/P008/P015/P016/P021; 15 rows', 'Use label fallback and record provenance', 'Recovers missing metadata without hiding where it came from.', 'firmware_normalized=2.1.8; source=label; raw blank retained.', 'Open'),
    ('T-03', 'Low', 'compare fallback firmware with populated versions', '15 fallbacks are 2.1.8; populated values are 2.2.0/2.2.1', 'Flag unexpected version; keep device row', 'The profile suggests a discrepancy but cannot prove the value is invalid.', '2.1.8 remains visible with review flag; no row discarded.', 'Open'),
    ('S-01', 'Medium', 'duration_min != whole-minute end-start', 'P007@2026-01-21 (4 sessions total)', 'Use duration_min; flag mismatch; retain interval evidence', 'Uses the documented metric and avoids silently rewriting source values.', 'Stage metric uses raw duration; quality rule records disagreement.', 'Implemented'),
    ('S-02', 'Medium', 'session span >12h or <3h', 'P026@2026-01-23 >12h; P010@2026-01-25 = 168 min', 'Flag long spans; review short spans', 'Anomaly flags preserve potentially valid sessions; short-span rule remains absent.', 'Long session carries SLEEP_SPAN_OUTLIER; short example remains visible for review.', 'Partial'),
    ('S-03', 'Medium', 'last stage start != sleep_end', 'P026@2026-01-23 (3 sessions)', 'Flag alignment issue; retain session', 'Surfaces a semantic/timing mismatch without dropping other sleep metrics.', 'Session stays in silver with SLEEP_FINAL_STAGE_MISALIGNED.', 'Implemented'),
    ('S-04', 'Low-Medium', 'supplied efficiency differs from stage-derived value', 'P001@2026-01-28; 218 sessions differ from span method', 'Derive from stages; preserve supplied value and flag mismatch', 'Stage timeline avoids the misleading near-100% span result.', 'Derived efficiency uses non-awake/all-stage minutes; supplied field remains available.', 'Implemented'),
    ('S-05', 'Low', 'sleep_onset has non-zero seconds', 'P001@2026-01-08; 1,379 sessions', 'Preserve timestamp and flag; do not round', 'Rounding could change a join key or create a minute collision.', 'Timestamp stays unchanged with SLEEP_ONSET_NOT_MINUTE_ALIGNED.', 'Implemented'),
    ('S-06', 'Low', 'compare stage sums with sleep_end-sleep_onset', 'P001@2026-01-08: all-stage sum exceeds span; only 7 non-awake sums differ', 'Use stage-based formulas; flag stage-level discrepancies', 'sleep_end marks wake-up, not necessarily the end of the final awake stage.', 'Awake duration is not counted as sleep; duration and gap flags remain.', 'Partial'),
    ('S-07', 'Medium', 'last stage end > sleep_end', 'P001@2026-01-08; 1,397 sessions have awake tail', 'Treat sleep_end as wake-up; exclude awake tail from sleep metrics', 'Avoids inflating sleep duration with post-wake time.', 'Gold total_sleep_min sums only non-awake stages.', 'Implemented'),
    ('M-01', 'High', "_unit == 'steps_per_hour'", 'P033 median 300 raw -> 5 per minute; 120,963 records', 'Convert non-negative multiples of 60 per record; preserve raw/unit', 'Evidence supports a rate, but provider confirmation is still absent; never infer unit by participant.', '300/hour -> normalized 5/min; raw 300 and unit text remain.', 'Implemented'),
    ('M-02', 'High', 'negative values and values >30 after unit conversion', 'P029/P037 include -1; max raw 1,000,000', 'Withhold negatives; classify >30/min as burst, retain in totals', 'Negative counts are invalid; positive bursts may be genuine and have no approved upper cap.', '-1 -> normalized NULL; 300/hour -> 5/min; 31/min -> burst and remains counted.', 'Implemented'),
    ('M-03', 'High', 'heart_rate_bpm outside inclusive 40-200', 'P029/P037 include -10 and 999 bpm; 4 records', 'Keep raw, withhold normalized HR, reject from HR aggregates', 'Prevents impossible readings contaminating HR while preserving unrelated fields.', '999 raw -> normalized NULL; daily HR excludes that slot.', 'Implemented'),
    ('M-04', 'Medium', '>=60 consecutive identical valid minute slots', 'P003@2026-01-09T04:44 and P042@2026-01-24T09:16; 180 min each', 'Flag whole run and withhold from HR aggregates; retain raw', 'Long flat runs may be stuck sensors; the 60-minute cutoff is provisional.', 'Each run contributes 0 flagged minutes to HR aggregate; raw values remain.', 'Implemented'),
    ('M-05', 'High', 'parse Z/offset as instant, convert to UTC-05:00', 'P018: 1,440 Z records; 600 records change local date', 'Convert timestamp; retain raw text and informational quality flag', 'Conversion fills empty grid slots and avoids collisions.', '04:00Z -> prior local date 23:00; all 2,880 Z records land on local grid.', 'Implemented'),
    ('M-06', 'Medium', 'count records per participant vs 40,320', 'P005=40,326; P020=39,960', 'Keep rows; expose valid counts/coverage', 'A count deviation signals incomplete or duplicate data, not that all records are unusable.', 'Daily valid-minute count reflects observed eligible slots; no blanket participant exclusion.', 'Partial'),
    ('M-07', 'High', 'group by participant-minute; compare normalized signatures', 'P019 duplicates 2026-01-10; steps 899 and HR 1,300 conflict records', 'Withhold conflicting slot; exact duplicates count once', 'No arbitrary first/last selection or averaging; avoids duplicate inflation.', 'Conflicting slot -> normalized value NULL; exact pair -> one aggregate contribution.', 'Implemented'),
    ('M-08', 'High', 'expected slots minus observed slots', 'P020@2026-01-27 is missing 360 minutes', 'Do not impute; expose coverage/valid count', 'Missing is not equivalent to zero activity or a measured HR value.', '1,080 observed valid slots stay usable; absent 360 slots remain missing.', 'Implemented'),
    ('M-09', 'Medium', 'timestamp seconds != 0 and slot collision check', 'P005 has 6 non-zero-second records; each shares a slot', 'Preserve timestamp, flag, then apply duplicate rule', 'Silent flooring changes source time and may merge distinct observations.', '00:01:30 remains stored; ambiguous slot is withheld if conflicting.', 'Implemented'),
    ('M-10', 'Low', 'compare each timestamp to prior file-order timestamp', 'One out-of-order record each in P010/P019/P022/P025', 'Sort for temporal processing; retain/flag source ordering', 'Chronological run checks require ordered timestamps; source ordering is still evidence.', 'Sorted working view can detect runs; original row/order remains preserved.', 'Open'),
    ('M-11', 'Low', 'compare steps and HR timestamp sequences per participant', 'P018 and P044 differ; 48/50 match', 'Expose stream-specific keys; do not assume positional alignment', 'Independent streams can be missing or shifted without sharing a common record index.', 'Join on normalized participant-minute keys, never row number.', 'Partial'),
    ('X-01', 'Medium', 'count observed minute slots per participant-local-date', 'P020@2026-01-27 has 360 missing slots', 'Keep partial day and report completeness/valid counts', 'Supports downstream thresholds without discarding measured slots.', 'Day reports 1,080 valid slots rather than claiming 1,440 or imputing.', 'Implemented'),
    ('X-02', 'Medium', 'union quality flags by participant-day', '108/1,400 flagged; P003 stuck run; P019 duplicate day', 'Persist per-measure/per-day flags', 'Consumers can filter affected measures without losing valid measures from that day.', 'Flagged HR can be omitted while valid steps/survey remain queryable.', 'Partial'),
    ('X-03', 'Low', 'compare onset with grid start; test HR coverage over session span', 'P003@2026-01-08 begins on 2026-01-07; 43 first-night sessions', 'Assign by wake date; keep the in-window session and report coverage', 'Overnight sessions belong to wake date; onset assignment would put 43 outside the study window.', 'Wake-date key remains 2026-01-08; pre-window onset is retained as context.', 'Partial'),
]

FINDING_COLUMNS = ('ID', 'Severity', 'How detected', 'Scenario example',
                   'Handling', 'Why', 'Solved example', 'Implementation status')
compact_rows('Finding register', [dict(zip(FINDING_COLUMNS, row)) for row in FINDINGS])
print(f'Finding IDs registered: {len(FINDINGS)}')

Finding register


ID,Severity,How detected,Scenario example,Handling,Why,Solved example,Implementation status
T-01,Medium,survey participant_id not in enrolled IDs,P999: 14 daily rows,Keep and flag in silver; exclude from enrolled outputs,Preserves source evidence without joining an unenrolled person.,"P999 rows remain auditable; enrolled gold stays at 1,400 rows.",Implemented
T-02,Medium,blank firmware_version with parseable device_label,P006/P008/P015/P016/P021; 15 rows,Use label fallback and record provenance,Recovers missing metadata without hiding where it came from.,firmware_normalized=2.1.8; source=label; raw blank retained.,Open
T-03,Low,compare fallback firmware with populated versions,15 fallbacks are 2.1.8; populated values are 2.2.0/2.2.1,Flag unexpected version; keep device row,The profile suggests a discrepancy but cannot prove the value is invalid.,2.1.8 remains visible with review flag; no row discarded.,Open
S-01,Medium,duration_min != whole-minute end-start,P007@2026-01-21 (4 sessions total),Use duration_min; flag mismatch; retain interval evidence,Uses the documented metric and avoids silently rewriting source values.,Stage metric uses raw duration; quality rule records disagreement.,Implemented
S-02,Medium,session span >12h or <3h,P026@2026-01-23 >12h; P010@2026-01-25 = 168 min,Flag long spans; review short spans,Anomaly flags preserve potentially valid sessions; short-span rule remains absent.,Long session carries SLEEP_SPAN_OUTLIER; short example remains visible for review.,Partial
S-03,Medium,last stage start != sleep_end,P026@2026-01-23 (3 sessions),Flag alignment issue; retain session,Surfaces a semantic/timing mismatch without dropping other sleep metrics.,Session stays in silver with SLEEP_FINAL_STAGE_MISALIGNED.,Implemented
S-04,Low-Medium,supplied efficiency differs from stage-derived value,P001@2026-01-28; 218 sessions differ from span method,Derive from stages; preserve supplied value and flag mismatch,Stage timeline avoids the misleading near-100% span result.,Derived efficiency uses non-awake/all-stage minutes; supplied field remains available.,Implemented
S-05,Low,sleep_onset has non-zero seconds,"P001@2026-01-08; 1,379 sessions",Preserve timestamp and flag; do not round,Rounding could change a join key or create a minute collision.,Timestamp stays unchanged with SLEEP_ONSET_NOT_MINUTE_ALIGNED.,Implemented
S-06,Low,compare stage sums with sleep_end-sleep_onset,P001@2026-01-08: all-stage sum exceeds span; only 7 non-awake sums differ,Use stage-based formulas; flag stage-level discrepancies,"sleep_end marks wake-up, not necessarily the end of the final awake stage.",Awake duration is not counted as sleep; duration and gap flags remain.,Partial
S-07,Medium,last stage end > sleep_end,"P001@2026-01-08; 1,397 sessions have awake tail",Treat sleep_end as wake-up; exclude awake tail from sleep metrics,Avoids inflating sleep duration with post-wake time.,Gold total_sleep_min sums only non-awake stages.,Implemented


Finding IDs registered: 24


## 3, 6-7. Typed source records and tabular/sleep evidence

Source values stay raw beside parsed dates, durations, and firmware candidates. These scans emit counts and participant/date examples only.

In [10]:
# 3, 6-7. Load typed record tables, then reproduce tabular and sleep checks.
PARTICIPANTS = read_csv_records(CFG.data_dir / 'participants' / 'participants.csv')
DEVICES = read_csv_records(CFG.data_dir / 'health_summaries' / 'device_metadata.csv')
SURVEY = read_csv_records(CFG.data_dir / 'health_summaries' / 'wellness_survey.csv')

# Raw text is retained; typed values live beside it for checks and joins.
DEVICE_ROWS = [
    {**row, 'firmware_version_raw': row.get('firmware_version', ''),
     'firmware_version_normalized': row.get('firmware_version', '').strip() or None}
    for row in DEVICES
]
SURVEY_ROWS = [
    {**row, 'local_date': parse_iso_date(row.get('date', '').strip()),
     'source_file': 'health_summaries/wellness_survey.csv', 'row_index': index}
    for index, row in enumerate(SURVEY, start=2)
]

# T-01: the orphan predicate is participant_id not in enrolled keys.
survey_orphans = [row for row in SURVEY_ROWS if row['participant_id'] not in ENROLLED]
# T-02/T-03: extract only a version token; never overwrite raw fields.
firmware_pattern = re.compile(r'fw([A-Za-z0-9.]+)', re.IGNORECASE)
firmware_fallbacks = []
for row in DEVICE_ROWS:
    match = firmware_pattern.search(row.get('device_label', ''))
    if not row['firmware_version_normalized'] and match:
        firmware_fallbacks.append((row['participant_id'], match.group(1)))

compact_rows('T findings reproduced from CSV', [
    finding_row('T-01', 'medium', 'flag + exclude from enrolled output', len(survey_orphans),
                sorted({row['participant_id'] for row in survey_orphans})),
    finding_row('T-02', 'medium', 'label fallback + provenance', len(firmware_fallbacks),
                [pid for pid, _ in firmware_fallbacks]),
    finding_row('T-03', 'low', 'review fallback value; retain device',
                sum(value == '2.1.8' for _, value in firmware_fallbacks),
                [pid for pid, value in firmware_fallbacks if value == '2.1.8']),
])

sleep_evidence: dict[str, list[str]] = defaultdict(list)
sleep_counts: Counter[str] = Counter()
for path in sorted((CFG.data_dir / 'wearable_events').glob('*/sleep.json')):
    for session in read_json_records(path):
        participant_id = session.get('participant_id', path.parent.name)
        session_date = session.get('date', '')
        example_id = f'{participant_id}@{session_date}'
        onset = local_timestamp(session.get('sleep_onset'))
        sleep_end = local_timestamp(session.get('sleep_end'))
        stages = session.get('stages') if isinstance(session.get('stages'), list) else []
        stage_intervals = []
        durations = []
        for stage in stages:
            start = local_timestamp(stage.get('start_time')) if isinstance(stage, dict) else None
            end = local_timestamp(stage.get('end_time')) if isinstance(stage, dict) else None
            duration = stage.get('duration_min') if isinstance(stage, dict) else None
            name = stage.get('stage') if isinstance(stage, dict) else None
            stage_intervals.append((name, start, end))
            if isinstance(duration, int) and not isinstance(duration, bool):
                durations.append((name, duration))
            if start and end and isinstance(duration, int) and not isinstance(duration, bool):
                if (end - start).total_seconds() != duration * 60:
                    sleep_counts['S-01'] += 1
                    if len(sleep_evidence['S-01']) < CFG.show_example_limit:
                        sleep_evidence['S-01'].append(example_id)
        span_min = ((sleep_end - onset).total_seconds() / 60
                    if onset and sleep_end else None)
        if span_min is not None:
            if span_min > 720:
                sleep_counts['S-02 >12h'] += 1
                if len(sleep_evidence['S-02 >12h']) < CFG.show_example_limit:
                    sleep_evidence['S-02 >12h'].append(example_id)
            if span_min < 180:
                sleep_counts['S-02 <3h'] += 1
                if len(sleep_evidence['S-02 <3h']) < CFG.show_example_limit:
                    sleep_evidence['S-02 <3h'].append(example_id)
        if stage_intervals and sleep_end and stage_intervals[-1][1] != sleep_end:
            sleep_counts['S-03'] += 1
            if len(sleep_evidence['S-03']) < CFG.show_example_limit:
                sleep_evidence['S-03'].append(example_id)
        asleep = sum(duration for name, duration in durations if name != 'awake')
        all_stage = sum(duration for _, duration in durations)
        supplied = session.get('efficiency_pct')
        if isinstance(supplied, (int, float)) and span_min and abs(supplied - asleep / span_min * 100) > 1:
            sleep_counts['S-04'] += 1
            if len(sleep_evidence['S-04']) < CFG.show_example_limit:
                sleep_evidence['S-04'].append(example_id)
        if onset and (onset.second or onset.microsecond):
            sleep_counts['S-05'] += 1
            if len(sleep_evidence['S-05']) < CFG.show_example_limit:
                sleep_evidence['S-05'].append(example_id)
        if span_min is not None and all_stage != span_min:
            sleep_counts['S-06 all stages'] += 1
            if len(sleep_evidence['S-06']) < CFG.show_example_limit:
                sleep_evidence['S-06'].append(example_id)
        if span_min is not None and asleep != span_min:
            sleep_counts['S-06 non-awake'] += 1
        if (stage_intervals and sleep_end and stage_intervals[-1][2]
                and stage_intervals[-1][2] > sleep_end):
            sleep_counts['S-07'] += 1
            if len(sleep_evidence['S-07']) < CFG.show_example_limit:
                sleep_evidence['S-07'].append(example_id)

sleep_rows = [
    {'finding_id': key, 'evidence_count': count,
     'example_ids': ', '.join(sleep_evidence.get(key.split()[0], [])[:CFG.show_example_limit])}
    for key, count in sorted(sleep_counts.items())
]
compact_rows('S findings reproduced from sleep JSON', sleep_rows)
print(f'Survey rows={len(SURVEY_ROWS)}; enrolled survey orphans={len(survey_orphans)}; '
      f'firmware fallbacks={len(firmware_fallbacks)}; sleep checks counted={sum(sleep_counts.values())}')


T findings reproduced from CSV
finding_id | severity | handling | evidence_count | example_ids
--- | --- | --- | --- | ---
T-01 | medium | flag + exclude from enrolled output | 14 | P999
T-02 | medium | label fallback + provenance | 15 | P006, P008, P015
T-03 | low | review fallback value; retain device | 15 | P006, P008, P015

S findings reproduced from sleep JSON
finding_id | evidence_count | example_ids
--- | --- | ---
S-01 | 4 | P007@2026-01-21, P010@2026-01-20, P024@2026-01-11
S-02 <3h | 2 | 
S-02 >12h | 3 | 
S-03 | 3 | P026@2026-01-23, P035@2026-01-20, P039@2026-01-14
S-04 | 218 | P001@2026-01-28, P002@2026-01-09, P002@2026-01-18
S-05 | 1379 | P001@2026-01-08, P001@2026-01-10, P001@2026-01-11
S-06 all stages | 1400 | P001@2026-01-08, P001@2026-01-09, P001@2026-01-10
S-06 non-awake | 7 | P001@2026-01-08, P001@2026-01-09, P001@2026-01-10
S-07 | 1397 | P001@2026-01-08, P001@2026-01-09, P001@2026-01-10
Survey rows=1414; enrolled survey orphans=14; firmware fallbacks=15; sleep checks

## 4, 8-9. Local minute keys, wearable, and cross-source evidence

For each participant, compare observed `(participant_id, local_date, minute_slot)` keys with 1,440 expected slots per day. UTC offsets are normalized before keys are made; completeness is counted sparsely rather than allocating a dense grid.

### Raw-label checks versus normalized-time checks

M-10 and M-11 are deliberately checked on source labels: ordering ignores explicit-offset rows, and stream-sequence comparison uses the raw timestamp strings. Grid, duplicate, stuck-run, and coverage checks use normalized local timestamps. Keeping these separate reproduces the original profiling semantics.

In [14]:
# 4, 8-9. Sparse local-minute keys; scan each participant file, never build a dense grid.
def profile_minute_source(source_name: str, value_key: str) -> dict[str, Any]:
    result: dict[str, Any] = {
        'records': 0, 'participants': {}, 'slots': {}, 'duplicates': 0,
        'conflict_slots': 0, 'nonaligned': 0, 'out_of_order': 0,
        'negative': 0, 'over_30_raw': 0, 'over_200_raw': 0,
        'over_1000_raw': 0, 'out_of_range': 0, 'unit_records': 0,
        'z_records': 0, 'z_date_changes': 0, 'stuck_runs': [],
        'sequence': {}, 'examples': defaultdict(list), 'daily_flags': defaultdict(set),
        'daily_slots': {}, 'duplicate_examples': [],
    }
    source_paths = sorted((CFG.data_dir / 'wearable_events').glob(f'*/{source_name}.json'))
    for path in source_paths:
        participant_id = path.parent.name
        rows = read_json_records(path)
        result['records'] += len(rows)
        result['participants'][participant_id] = len(rows)
        slot_values: dict[datetime, list[Any]] = defaultdict(list)
        day_slots: dict[date, set[datetime]] = defaultdict(set)
        raw_sequence = []
        previous_naive_slot = None
        out_of_order_for_file = 0
        for row in rows:
            raw_timestamp = row.get('timestamp')
            raw_sequence.append(raw_timestamp)
            timestamp = local_timestamp(raw_timestamp)
            value = row.get(value_key)
            if source_name == 'steps':
                if row.get('_unit') == 'steps_per_hour':
                    result['unit_records'] += 1
                    if timestamp:
                        result['daily_flags'][(participant_id, timestamp.date())].add('unit')
                if isinstance(value, int) and not isinstance(value, bool):
                    result['negative'] += value < 0
                    result['over_30_raw'] += value > 30
                    result['over_200_raw'] += value > 200
                    result['over_1000_raw'] += value > 1000
                    if value < 0 and timestamp:
                        result['daily_flags'][(participant_id, timestamp.date())].add('negative')
            elif isinstance(value, int) and not isinstance(value, bool):
                outside = not CFG.heart_rate_min <= value <= CFG.heart_rate_max
                result['out_of_range'] += outside
                if outside and timestamp:
                    result['daily_flags'][(participant_id, timestamp.date())].add('range')
            if timestamp is None:
                continue
            if isinstance(raw_timestamp, str) and raw_timestamp.endswith('Z'):
                result['z_records'] += 1
                try:
                    if timestamp.date() != date.fromisoformat(raw_timestamp[:10]):
                        result['z_date_changes'] += 1
                except ValueError:
                    pass
            if timestamp.second or timestamp.microsecond:
                result['nonaligned'] += 1
                if len(result['examples']['nonaligned']) < CFG.show_example_limit:
                    result['examples']['nonaligned'].append(participant_id)
            explicit_timezone = isinstance(raw_timestamp, str) and (
                raw_timestamp.endswith('Z') or re.search(r'[+-][0-9]{2}:[0-9]{2}$', raw_timestamp)
            )
            source_slot = minute_slot(timestamp)
            if not explicit_timezone and CFG.study_start <= source_slot.date() <= CFG.study_dates[-1]:
                if previous_naive_slot is not None and source_slot < previous_naive_slot:
                    out_of_order_for_file += 1
                    if participant_id not in result['examples']['out_of_order']:
                        result['examples']['out_of_order'].append(participant_id)
                previous_naive_slot = source_slot
            slot_values[source_slot].append(value)
            day_slots[timestamp.date()].add(source_slot)
        result['out_of_order'] += out_of_order_for_file
        result['sequence'][participant_id] = tuple(raw_sequence)
        result['daily_slots'][participant_id] = {
            day: len(day_slots.get(day, set())) for day in CFG.study_dates
        }
        for slot, values in slot_values.items():
            if len(values) > 1:
                result['duplicates'] += len(values) - 1
                conflict = len({json.dumps(value, sort_keys=True) for value in values}) > 1
                if conflict:
                    result['conflict_slots'] += 1
                    if len(result['duplicate_examples']) < CFG.show_example_limit:
                        result['duplicate_examples'].append(
                            f'{participant_id}@{slot.isoformat(timespec="minutes")}')
                    result['daily_flags'][(participant_id, slot.date())].add('duplicate')
        if source_name == 'heart_rate':
            clean_slots = []
            for slot, values in slot_values.items():
                if len(set(values)) == 1 and isinstance(values[0], int) \
                        and CFG.heart_rate_min <= values[0] <= CFG.heart_rate_max:
                    clean_slots.append((slot, values[0]))
            clean_slots.sort()
            run = []
            for item in [*clean_slots, None]:
                if run and item and item[1] == run[-1][1] \
                        and item[0] - run[-1][0] == timedelta(minutes=1):
                    run.append(item)
                    continue
                if len(run) >= CFG.stuck_minutes:
                    result['stuck_runs'].append((participant_id, run[0][0], len(run)))
                    for run_slot, _ in run:
                        result['daily_flags'][(participant_id, run_slot.date())].add('stuck')
                run = [item] if item else []
    result['missing_slots'] = sum(
        CFG.expected_minutes_per_day - count
        for days in result['daily_slots'].values() for count in days.values()
    )
    result['complete_days'] = sum(
        count == CFG.expected_minutes_per_day
        for days in result['daily_slots'].values() for count in days.values()
    )
    result['zero_days'] = sum(
        count == 0 for days in result['daily_slots'].values() for count in days.values()
    )
    result['partial_days'] = sum(
        0 < count < CFG.expected_minutes_per_day
        for days in result['daily_slots'].values() for count in days.values()
    )
    return result


STEPS_PROFILE = profile_minute_source('steps', 'steps')
HR_PROFILE = profile_minute_source('heart_rate', 'heart_rate_bpm')
sequence_differences = [
    participant_id for participant_id in sorted(ENROLLED)
    if STEPS_PROFILE['sequence'].get(participant_id) != HR_PROFILE['sequence'].get(participant_id)
]
flagged_days = {
    key for profile in (STEPS_PROFILE, HR_PROFILE) for key, flags in profile['daily_flags'].items() if flags
}

minute_findings = [
    finding_row('M-01', 'high', 'per-record unit check', STEPS_PROFILE['unit_records'], ['P005', 'P033', 'P038']),
    finding_row('M-02 negative', 'high', 'negative steps', STEPS_PROFILE['negative'], ['P029', 'P037']),
    finding_row('M-02 >30 raw', 'high', 'raw values above expected rate', STEPS_PROFILE['over_30_raw'], ['P029', 'P037']),
    finding_row('M-03', 'high', 'HR outside inclusive range', HR_PROFILE['out_of_range'], ['P029', 'P037']),
    finding_row('M-04', 'medium', 'identical in-range runs >=60 slots', len(HR_PROFILE['stuck_runs']),
                [f'{pid}@{start.isoformat(timespec="minutes")}' for pid, start, _ in HR_PROFILE['stuck_runs']]),
    finding_row('M-05', 'high', 'explicit UTC Z timestamps', HR_PROFILE['z_records'], ['P018', 'P044']),
    finding_row('M-06 steps', 'medium', 'participant record counts',
                sum(n != 40320 for n in STEPS_PROFILE['participants'].values()), ['P005', 'P020']),
    finding_row('M-07 steps', 'high', 'duplicate slots with differing values', STEPS_PROFILE['conflict_slots'],
                STEPS_PROFILE['duplicate_examples']),
    finding_row('M-08 steps', 'high', 'missing local minute slots', STEPS_PROFILE['missing_slots'], ['P010', 'P020']),
    finding_row('M-09 steps', 'medium', 'timestamps with seconds/microseconds', STEPS_PROFILE['nonaligned'],
                STEPS_PROFILE['examples']['nonaligned']),
    finding_row('M-10 steps', 'low', 'timestamps earlier than preceding in-window naive minute',
                STEPS_PROFILE['out_of_order'], STEPS_PROFILE['examples']['out_of_order']),
    finding_row('M-11', 'low', 'steps/HR raw timestamp sequence mismatch',
                len(sequence_differences), sequence_differences),
]
compact_rows('M findings reproduced by streaming participant files', minute_findings)

print(f"X-01 full local-grid days: steps={STEPS_PROFILE['complete_days']}/1400; "
      f"heart_rate={HR_PROFILE['complete_days']}/1400")
print(f"X-01 partial/zero days: steps={STEPS_PROFILE['partial_days']}/{STEPS_PROFILE['zero_days']}; "
      f"heart_rate={HR_PROFILE['partial_days']}/{HR_PROFILE['zero_days']}")
print(f'X-02 distinct participant-days with at least one quality signal: {len(flagged_days)}')
print('X-03 first-night sessions before grid start: 43 (wake-date assignment retains them)')


M findings reproduced by streaming participant files
finding_id | severity | handling | evidence_count | example_ids
--- | --- | --- | --- | ---
M-01 | high | per-record unit check | 120963 | P005, P033, P038
M-02 negative | high | negative steps | 2 | P029, P037
M-02 >30 raw | high | raw values above expected rate | 95902 | P029, P037
M-03 | high | HR outside inclusive range | 4 | P029, P037
M-04 | medium | identical in-range runs >=60 slots | 2 | P003@2026-01-09T04:44, P042@2026-01-24T09:16
M-05 | high | explicit UTC Z timestamps | 2880 | P018, P044
M-06 steps | medium | participant record counts | 8 | P005, P020
M-07 steps | high | duplicate slots with differing values | 3590 | P005@2026-01-10T13:55, P005@2026-01-14T08:41, P005@2026-01-14T08:42
M-08 steps | high | missing local minute slots | 6840 | P010, P020
M-09 steps | medium | timestamps with seconds/microseconds | 28 | P005, P005, P005
M-10 steps | low | timestamps earlier than preceding in-window naive minute | 4 | P010, P01

In [ ]:
# Concrete IDs for sparse evidence cases not all shown in the aggregate count table.
compact_rows('Sleep and cross-source scenario examples', [
    finding_row('S-02 >12h', 'medium', 'flag for review', sleep_counts['S-02 >12h'],
                sleep_evidence['S-02 >12h']),
    finding_row('S-02 <3h', 'medium', 'review; no implemented short-span rule', sleep_counts['S-02 <3h'],
                sleep_evidence['S-02 <3h']),
    finding_row('S-03', 'medium', 'flag final-stage alignment', sleep_counts['S-03'], sleep_evidence['S-03']),
    finding_row('S-06', 'low', 'stage timeline; exclude awake from sleep',
                sleep_counts['S-06 all stages'], sleep_evidence['S-06']),
    finding_row('X-03', 'low', 'wake-date assignment', 43,
                ['P003@2026-01-08', 'P001@2026-01-08']),
])

## 10-13. Scenario walkthroughs and solved examples

Each demonstration below starts from a real, identified scenario, calls the production silver-layer function where one exists, and reports only the relevant before/after fields. A small synthetic case is used for unsupported-unit behavior. The source files are read-only; no pipeline tables or raw files are modified.

In [19]:
# 10. Unit conversion through the production steps layer.
def event_path(participant_id: str, source: str) -> Path:
    return CFG.data_dir / 'wearable_events' / participant_id / f'{source}.json'


def relative_event_path(participant_id: str, source: str) -> str:
    return f'wearable_events/{participant_id}/{source}.json'


p033_rows = read_json_records(event_path('P033', 'steps'))
p033_hourly = next(row for row in p033_rows
                   if row.get('_unit') == 'steps_per_hour' and row.get('steps', 0) >= 120)
p033_solved = build_steps_layer(
    {relative_event_path('P033', 'steps'): [p033_hourly]}, ENROLLED, CFG.study_dates
).rows[0]
compact_rows('M-01 before -> after', [{
    'participant': 'P033', 'raw_steps': p033_solved[3], 'raw_unit': p033_solved[6],
    'normalized_steps_per_minute': p033_solved[4], 'flag': p033_solved[10],
}])

# Unsupported and negative values are withheld; values over 30/min remain as burst activity.
def one_step(value: int, unit: str | None = None):
    raw = {'participant_id': 'P029', 'timestamp': '2026-01-08T12:00:00', 'steps': value}
    if unit is not None:
        raw['_unit'] = unit
    return build_steps_layer(
        {relative_event_path('P029', 'steps'): [raw]}, ENROLLED, CFG.study_dates
    ).rows[0]

unsupported, negative, burst = one_step(61, 'steps_per_hour'), one_step(-1), one_step(31)
compact_rows('M-01/M-02 edge cases', [
    {'case': '61 steps/hour', 'normalized': unsupported[4], 'flag': unsupported[10]},
    {'case': '-1 step', 'normalized': negative[4], 'flag': negative[10]},
    {'case': '31 steps/min', 'normalized': burst[4], 'activity_level': burst[5], 'retained': True},
])

# 11. Conflict, exact duplicate and missing-grid examples through silver/gold layer outputs.
def local_slot_values(participant_id: str, source: str, value_key: str):
    by_slot: dict[datetime, list[Any]] = defaultdict(list)
    for raw in read_json_records(event_path(participant_id, source)):
        timestamp = local_timestamp(raw.get('timestamp'))
        if timestamp:
            by_slot[minute_slot(timestamp)].append(raw.get(value_key))
    return by_slot


p019_rows = read_json_records(event_path('P019', 'steps'))
p019_layer = build_steps_layer(
    {relative_event_path('P019', 'steps'): p019_rows}, ENROLLED, CFG.study_dates
)
p019_slots = local_slot_values('P019', 'steps', 'steps')
conflict_slot, conflict_values = next(
    (slot, values) for slot, values in p019_slots.items()
    if slot.date() == date(2026, 1, 10) and len(values) > 1 and len(set(values)) > 1
)
conflict_silver = next(
    row for row in p019_layer.rows
    if minute_slot(row[2]) == conflict_slot and 'STEPS_DUPLICATE_MINUTE_CONFLICT' in (row[10] or '')
)
p050_rows = read_json_records(event_path('P050', 'steps'))
p050_layer = build_steps_layer(
    {relative_event_path('P050', 'steps'): p050_rows}, ENROLLED, CFG.study_dates
)
p050_slots = local_slot_values('P050', 'steps', 'steps')
exact_slot, exact_values = next(
    (slot, values) for slot, values in p050_slots.items()
    if len(values) > 1 and len(set(values)) == 1
)
exact_silver = [row for row in p050_layer.rows
                if minute_slot(row[2]) == exact_slot
                and 'STEPS_DUPLICATE_MINUTE_EXACT' in (row[10] or '')]
p020_rows = read_json_records(event_path('P020', 'steps'))
p020_layer = build_steps_layer(
    {relative_event_path('P020', 'steps'): p020_rows}, ENROLLED, CFG.study_dates
)
p020_day = p020_layer.daily[('P020', date(2026, 1, 27))]
compact_rows('M-07/M-08 before -> after', [
    {'case': 'P019 conflict on 2026-01-10', 'slot': conflict_slot,
     'distinct_raw_values': len(set(conflict_values)), 'normalized_value': conflict_silver[4],
     'quality_flag': conflict_silver[10]},
    {'case': 'P050 exact duplicate', 'slot': exact_slot, 'source_rows': len(exact_silver),
     'normalized_value_counted_once': exact_silver[0][4]},
    {'case': 'P020 partial day 2026-01-27', 'valid_minutes': p020_day.steps_valid_minute_count,
     'missing_minutes': 1440 - p020_day.steps_valid_minute_count, 'imputed': 0},
])

# 12. HR range, stuck-run, and UTC conversion handling through the production HR layer.
hr_layers = {}
for participant_id in ('P029', 'P003', 'P018'):
    hr_rows = read_json_records(event_path(participant_id, 'heart_rate'))
    hr_layers[participant_id] = build_heart_rate_layer(
        {relative_event_path(participant_id, 'heart_rate'): hr_rows}, ENROLLED, CFG.study_dates
    )
p029_layer, p003_layer, p018_layer = (hr_layers[pid] for pid in ('P029', 'P003', 'P018'))
invalid_hr = next(row for row in p029_layer.rows if row[9])
stuck_rows = [row for row in p003_layer.rows if row[10]]
converted_hr = next(row for row in p018_layer.rows
                    if row[11] and row[2].date() != date.fromisoformat(row[3][:10]))
compact_rows('M-03/M-04/M-05 before -> after', [
    {'case': 'P029 invalid HR', 'raw_bpm': invalid_hr[4], 'normalized_bpm': invalid_hr[5],
     'flag': invalid_hr[12]},
    {'case': 'P003 stuck run', 'flagged_minutes': len(stuck_rows),
     'normalized_values_withheld': sum(row[5] is None for row in stuck_rows)},
    {'case': 'P018 UTC record', 'raw_timestamp': converted_hr[3],
     'local_timestamp': converted_hr[2], 'flag': converted_hr[12]},
])

# 13. Sleep silver/gold example; the production metric excludes the awake tail.
sleep_ids = ('P007', 'P026', 'P035', 'P039')
sleep_files = {
    relative_event_path(pid, 'sleep'): read_json_records(event_path(pid, 'sleep'))
    for pid in sleep_ids
}
sleep_layer = build_sleep_layer(sleep_files, ENROLLED, CFG.study_dates)
p007_raw = next(row for row in sleep_files[relative_event_path('P007', 'sleep')]
                if row.get('date') == '2026-01-21')
p007_metrics = compute_sleep_metrics([
    (stage.get('stage', ''), stage.get('duration_min')) for stage in p007_raw['stages']
])
compact_rows('S-01/S-04/S-07 before -> after', [{
    'scenario': 'P007@2026-01-21', 'stage_metric_total_sleep_min': p007_metrics.total_sleep_min,
    'derived_efficiency_pct': round(p007_metrics.sleep_efficiency_pct, 1),
    'mismatch_flags_in_selected_sessions': sleep_layer.failed['SLEEP_STAGE_DURATION_MISMATCH'],
    'awake_tail_counted_as_sleep': False,
}])

# Tabular/cross-source results are decision examples; firmware fallback remains unimplemented.
firmware_example = next(row for row in DEVICE_ROWS if row['participant_id'] == 'P006')
firmware_match = firmware_pattern.search(firmware_example.get('device_label', ''))
compact_rows('T-01/T-02/X-01/X-03 field-level outcomes', [
    {'scenario': 'P999 survey orphan', 'silver_flag': 'SURVEY_ORPHAN_PARTICIPANT',
     'raw_retained': True, 'included_in_enrolled_gold': False},
    {'scenario': 'P006 firmware fallback', 'raw_blank_retained': True,
     'candidate': firmware_match.group(1) if firmware_match else None,
     'provenance': 'device_label', 'status': 'open'},
    {'scenario': 'P020@2026-01-27', 'valid_minutes': p020_day.steps_valid_minute_count,
     'missing_minutes': 360, 'imputed': 0},
    {'scenario': 'P003@2026-01-08 first-night sleep', 'wake_date_key': '2026-01-08',
     'onset_before_grid': True, 'session_retained': True},
])

M-01 before -> after


participant,raw_steps,raw_unit,normalized_steps_per_minute,flag
P033,120,steps_per_hour,2,STEPS_UNIT_CONVERTED


M-01/M-02 edge cases


case,normalized,flag,activity_level,retained
61 steps/hour,None,STEPS_UNIT_UNSUPPORTED,,
-1 step,None,STEPS_NEGATIVE,,
31 steps/min,31,,burst,True


M-07/M-08 before -> after


...

case,raw_bpm,normalized_bpm,flag,flagged_minutes,normalized_values_withheld,raw_timestamp,local_timestamp
P029 invalid HR,-10,None,HR_VALUE_OUT_OF_RANGE,,,,
P003 stuck run,,,,180,180,,
P018 UTC record,,,HR_TIMESTAMP_CONVERTED,,,2026-01-26T00:00:00Z,2026-01-25 19:00:00


S-01/S-04/S-07 before -> after


scenario,stage_metric_total_sleep_min,derived_efficiency_pct,mismatch_flags_in_selected_sessions,awake_tail_counted_as_sleep
P007@2026-01-21,523,88.3,2,False


T-01/T-02/X-01/X-03 field-level outcomes


scenario,silver_flag,raw_retained,included_in_enrolled_gold,raw_blank_retained,candidate,provenance,status,valid_minutes,missing_minutes,imputed,wake_date_key,onset_before_grid,session_retained
P999 survey orphan,SURVEY_ORPHAN_PARTICIPANT,True,False,,,,,,,,,,
P006 firmware fallback,,,,True,2.1.8,device_label,open,,,,,,
P020@2026-01-27,,,,,,,,1080,360,0,,,
P003@2026-01-08 first-night sleep,,,,,,,,,,,2026-01-08,True,True


## 14-15. Validation and rule-helper tests

These checks fail loudly if core profile counts or handling examples drift. The reference values are profiling evidence, not thresholds inferred by the notebook. The full pipeline run remains separately covered by INC-002 through INC-005 tests and review notebooks.

In [21]:
# Core profiling reference counts. Any assertion failure means inspect the source check before changing a rule.
expected = {
    'enrolled participants': (len(ENROLLED), 50),
    'survey rows': (len(SURVEY_ROWS), 1414),
    'survey orphans': (len(survey_orphans), 14),
    'firmware fallbacks': (len(firmware_fallbacks), 15),
    'firmware 2.1.8 fallbacks': (sum(value == '2.1.8' for _, value in firmware_fallbacks), 15),
    'S-01 duration mismatches': (sleep_counts['S-01'], 4),
    'S-02 long spans': (sleep_counts['S-02 >12h'], 3),
    'S-02 short spans': (sleep_counts['S-02 <3h'], 2),
    'S-03 final-stage starts': (sleep_counts['S-03'], 3),
    'S-04 efficiency disagreements': (sleep_counts['S-04'], 218),
    'S-05 non-minute-aligned onsets': (sleep_counts['S-05'], 1379),
    'S-06 all-stage sum/span differences': (sleep_counts['S-06 all stages'], 1400),
    'S-06 non-awake sum/span differences': (sleep_counts['S-06 non-awake'], 7),
    'S-07 awake tails': (sleep_counts['S-07'], 1397),
    'M-01 hourly-unit records': (STEPS_PROFILE['unit_records'], 120963),
    'M-02 negative steps': (STEPS_PROFILE['negative'], 2),
    'M-02 raw steps >30': (STEPS_PROFILE['over_30_raw'], 95902),
    'M-03 out-of-range HR': (HR_PROFILE['out_of_range'], 4),
    'M-04 stuck HR runs': (len(HR_PROFILE['stuck_runs']), 2),
    'M-04 stuck HR minutes': (sum(length for _, _, length in HR_PROFILE['stuck_runs']), 360),
    'M-05 Z timestamps': (HR_PROFILE['z_records'], 2880),
    'M-05 Z local-date changes': (HR_PROFILE['z_date_changes'], 600),
    'M-07 steps excess duplicate records': (STEPS_PROFILE['duplicates'], 5790),
    'M-07 HR excess duplicate records': (HR_PROFILE['duplicates'], 5790),
    'M-08 steps missing local slots': (STEPS_PROFILE['missing_slots'], 6840),
    'M-08 HR missing local slots after conversion': (HR_PROFILE['missing_slots'], 6840),
    'M-09 steps non-aligned timestamps': (STEPS_PROFILE['nonaligned'], 28),
    'M-09 HR non-aligned timestamps': (HR_PROFILE['nonaligned'], 28),
    'M-10 steps out-of-order records': (STEPS_PROFILE['out_of_order'], 4),
    'M-10 HR out-of-order records': (HR_PROFILE['out_of_order'], 4),
    'M-11 stream sequence differences': (len(sequence_differences), 2),
    'X-01 complete steps participant-days': (STEPS_PROFILE['complete_days'], 1393),
    'X-01 complete HR participant-days': (HR_PROFILE['complete_days'], 1393),
}

# X-02 counts any duplicate slot (exact or conflicting), plus the other approved quality signals.
def duplicate_days(source_name: str, value_key: str) -> set[tuple[str, date]]:
    found = set()
    for path in sorted((CFG.data_dir / 'wearable_events').glob(f'*/{source_name}.json')):
        participant_id = path.parent.name
        by_slot: dict[datetime, int] = Counter()
        for row in read_json_records(path):
            timestamp = local_timestamp(row.get('timestamp'))
            if timestamp and row.get(value_key) is not None:
                by_slot[minute_slot(timestamp)] += 1
        found.update((participant_id, slot.date()) for slot, count in by_slot.items() if count > 1)
    return found


flagged_days.update(duplicate_days('steps', 'steps'))
flagged_days.update(duplicate_days('heart_rate', 'heart_rate_bpm'))
expected['X-02 distinct quality-flagged days'] = (len(flagged_days), 108)

first_night_count = 0
first_night_examples = []
for path in sorted((CFG.data_dir / 'wearable_events').glob('*/sleep.json')):
    for session in read_json_records(path):
        onset = local_timestamp(session.get('sleep_onset'))
        if onset and onset.date() < CFG.study_start:
            first_night_count += 1
            if len(first_night_examples) < CFG.show_example_limit:
                first_night_examples.append(f"{session.get('participant_id', path.parent.name)}@{session.get('date')}")
expected['X-03 first-night sessions'] = (first_night_count, 43)

results = []
for label, (observed, target) in expected.items():
    results.append({'check': label, 'observed': observed, 'expected': target,
                    'status': 'PASS' if observed == target else 'FAIL'})
compact_rows('Profile regression checks', results)
assert all(observed == target for observed, target in expected.values()), 'Profile reference mismatch'

# 15. Fast edge tests for the shared helpers and production-layer decisions.
test_timestamp = local_timestamp('2026-01-09T04:00:00Z')
assert test_timestamp == datetime(2026, 1, 8, 23, 0)
assert minute_slot(datetime(2026, 1, 8, 23, 1, 59)) == datetime(2026, 1, 8, 23, 1)
test_dups = duplicate_groups([
    ('P001', datetime(2026, 1, 8, 12, 0), 3),
    ('P001', datetime(2026, 1, 8, 12, 0, 30), 3),
    ('P001', datetime(2026, 1, 8, 12, 0, 45), 4),
])
assert len(test_dups) == 1 and test_dups[('P001', datetime(2026, 1, 8, 12, 0))] == [3, 3, 4]
assert p033_solved[4] * 60 == p033_solved[3]
assert unsupported[4] is None and 'STEPS_UNIT_UNSUPPORTED' in unsupported[10]
assert negative[4] is None and 'STEPS_NEGATIVE' in negative[10]
assert burst[4] == 31 and burst[5] == 'burst'
assert invalid_hr[4] not in range(CFG.heart_rate_min, CFG.heart_rate_max + 1)
assert invalid_hr[5] is None
assert len(stuck_rows) == 180 and all(row[5] is None for row in stuck_rows)
assert converted_hr[11] and converted_hr[3].endswith('Z')
metric_test = compute_sleep_metrics([('light', 80), ('deep', 20), ('awake', 10)])
assert metric_test.total_sleep_min == 100
assert round(metric_test.deep_sleep_pct, 1) == 20.0
assert round(metric_test.sleep_efficiency_pct, 1) == 90.9
print('\nRule-helper and production-layer edge tests: PASS (timezone, slot grouping, unit conversion, range, stuck run, sleep formulas)')
print(f'X-03 examples: {", ".join(first_night_examples)}')

Profile regression checks


check,observed,expected,status
enrolled participants,50,50,PASS
survey rows,1414,1414,PASS
survey orphans,14,14,PASS
firmware fallbacks,15,15,PASS
firmware 2.1.8 fallbacks,15,15,PASS
S-01 duration mismatches,4,4,PASS
S-02 long spans,3,3,PASS
S-02 short spans,2,2,PASS
S-03 final-stage starts,3,3,PASS
S-04 efficiency disagreements,218,218,PASS



Rule-helper and production-layer edge tests: PASS (timezone, slot grouping, unit conversion, range, stuck run, sleep formulas)
X-03 examples: P001@2026-01-08, P002@2026-01-08, P004@2026-01-08


In [22]:
# Final registry completeness and exact-duplicate aggregation check.
expected_ids = {f'T-{n:02d}' for n in range(1, 4)}
expected_ids |= {f'S-{n:02d}' for n in range(1, 8)}
expected_ids |= {f'M-{n:02d}' for n in range(1, 12)}
expected_ids |= {f'X-{n:02d}' for n in range(1, 4)}
registered_ids = {row[0] for row in FINDINGS}
assert registered_ids == expected_ids, f'Registry IDs differ: {registered_ids ^ expected_ids}'
assert len(exact_silver) >= 2
assert len({row[4] for row in exact_silver}) == 1
print(f'Registry completeness: PASS ({len(registered_ids)} finding IDs)')
print(f'P050 exact duplicate: {len(exact_silver)} raw rows, one unique minute/value contribution.')
print('Notebook scope: profiling + layer-level scenarios; no output materialized to disk.')

Registry completeness: PASS (24 finding IDs)
P050 exact duplicate: 3 raw rows, one unique minute/value contribution.
Notebook scope: profiling + layer-level scenarios; no output materialized to disk.
